# Проверка опыта по требованиям вакансии

Я начал с простого условия: нужные признаки должны быть в одной исходной фразе. Для каждого требования есть своя карточка. Общий граф доменов я не строю. Эмбеддинги помогают найти похожий опыт, затем карточка проверяет, хватает ли фактов.

В SQL 117 кандидатов и 695 упоминаний. Полных резюме и проектов в нём нет.

In [1]:
from pathlib import Path
import sys, os, json
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from data import load_candidates
from matcher import load_cards, check_phrase, rank_candidates
from experiment import run
rows, audit = load_candidates(ROOT / 'data/domains.sql')
cards = load_cards(ROOT / 'cards.json')
print(audit)

{'rows': 117, 'mentions': 695, 'unique_phrases': 541, 'repaired_rows': 117, 'source_sha256': '7128e15c6ca1b6f392c799d8085c629c368ef38a7ca5d1b9f78d8c6cb1a761c5'}


## Как работает карточка

Для интеграции банковского ядра нужны три признака: действие, объект и банковский контекст. Одно слово про банк не подтверждает работу с ядром. Карточка хранит варианты формулировок только для своего требования. Её можно менять, не перестраивая общий справочник.

In [2]:
core = cards['bank_core_integration']
print('Проверяем:', core['text'])
print('Нужны признаки:', list(core['groups']))
by_id = {r['id']: r['domains'] for r in rows}
for candidate_id in (79, 6, 50, 83):
    result = rank_candidates([{'id': candidate_id, 'domains': by_id[candidate_id]}],
                             [{'card': 'bank_core_integration', 'required': True}], cards)[0]
    check = result['checks'][0]
    print(candidate_id, check['status'], check['quote'], 'не хватает:', check['missing'])

Проверяем: Опыт интеграции банковского ядра
Нужны признаки: ['интеграция', 'ядро или АБС', 'банковский контекст']
79 confirmed интеграция банковского ядра не хватает: []
6 review банковские интеграции не хватает: ['ядро или АБС']
50 review интеграция банковских систем не хватает: ['ядро или АБС']
83 review банковский сектор не хватает: ['интеграция', 'ядро или АБС']


## Примеры, которые легко перепутать

Следующие фразы придуманы для проверки правил. В исходном SQL нет отрицаний и описаний отдельных проектов. Проверка нескольких таких фраз не заменяет разбор настоящего резюме.

In [3]:
from matcher import check_candidate
for text in ['интеграцией банковского ядра не занимался',
             'интеграция ABS автомобиля', 'тестирование АБС банка']:
    print(text, check_phrase(text, core)['status'])
split = [{'text': 'интеграция CRM', 'project_id': 'A'},
         {'text': 'ядро банковской системы', 'project_id': 'B'}]
print('Разные проекты:', check_candidate(split, core)['status'])

интеграцией банковского ядра не занимался contradicted
интеграция ABS автомобиля review
тестирование АБС банка review
Разные проекты: review


## Сравнение на данных задания

Сравниваю три варианта: поиск по словам, локальные эмбеддинги и проверку по карточке. В последнем варианте эмбеддинги задают порядок только при равном числе подтверждений. Сама похожесть не закрывает обязательное условие.

В checks.json есть 10 авторских сценариев и 34 оценки выбранных пар. Они нужны для начальной проверки. Это не независимая экспертная разметка всей базы. nDCG@2 считается только среди тех кандидатов, которым заранее дали оценку.

In [4]:
cache = Path(os.environ.get('DOMAIN_CACHE', str(ROOT / '.cache')))
report = run(cache=cache, benchmark=10000)
print('Средняя nDCG@2:')
for method, value in report['methods'].items():
    print(method, round(value, 3))

C:\Users\Administrator\Desktop\нейрокоманда\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "данные": {
    "rows": 117,
    "mentions": 695,
    "unique_phrases": 541,
    "repaired_rows": 117,
    "source_sha256": "7128e15c6ca1b6f392c799d8085c629c368ef38a7ca5d1b9f78d8c6cb1a761c5"
  },
  "nDCG@2": {
    "Поиск по словам": 0.9568332785638866,
    "Эмбеддинги фраз": 0.9181479978404324,
    "Проверка по карточке": 0.982623465712856
  },
  "модель": true
}
Средняя nDCG@2:
Поиск по словам 0.957
Эмбеддинги фраз 0.918
Проверка по карточке 0.983


## Запрос из задания

Финтех обязателен, интеграция ядра желательна. Сначала идут кандидаты, у которых есть все обязательные подтверждения. Затем учитывается желательный опыт. Балла вероятности найма здесь нет.

In [5]:
for candidate in report['demo'][:3]:
    print('Кандидат', candidate['id'], 'обязательные:', candidate['required_confirmed'],
          'желательные:', candidate['desired_confirmed'])
    for check in candidate['checks']:
        print(' ', check['text'], check['status'], check['quote'])
print(report['timing'])

Кандидат 79 обязательные: 1 желательные: 1
  Опыт в финтехе confirmed банковский сектор
  Опыт интеграции банковского ядра confirmed интеграция банковского ядра
Кандидат 6 обязательные: 1 желательные: 0
  Опыт в финтехе confirmed финтех
  Опыт интеграции банковского ядра review банковские интеграции
Кандидат 59 обязательные: 1 желательные: 0
  Опыт в финтехе confirmed финтех
  Опыт интеграции банковского ядра review интеграция банковских сервисов
{'model_or_cache_load_seconds': 0.7069060000012541, 'repeat_profiles': {'rows': 10000, 'seconds': 2.512564900000143, 'note': 'Повтор исходных 117 профилей. Замер проверки карточек без извлечения и расчёта эмбеддингов.'}}


## Что я из этого понял

На этих коротких фразах карточки помогают отделить подтверждение от общей тематики. Но правила пока написаны вручную. Новое требование тоже требует своей карточки. Следующий шаг я бы проверял на полных резюме: извлечение цитат через LLM и проверку спорных пар с учётом проекта. В текущем опыте LLM не запускалась.

Для 10 000 кандидатов разумно заранее хранить фразы и векторы. Здесь замер повторяет исходные 117 профилей, поэтому он показывает только время проверки карточек. Он не измеряет обработку 10 000 новых резюме.

Модель: [multilingual MiniLM](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). Она работает локально на CPU. Веса и их версия указаны в results/experiment.json.